In [4]:
!nvidia-smi

Fri May  1 05:13:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install -q diffusers transformers accelerate peft datasets bitsandbytes safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00


In [6]:
!pip install -q xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 44.8 MB/s eta 0:00:00


In [12]:
from huggingface_hub import notebook_login
notebook_login()

In [15]:
import torch
from diffusers import StableDiffusionPipeline
from diffusers import DDPMScheduler
from diffusers import UNet2DConditionModel
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
from PIL import Image
import os

In [47]:
from google.colab import files
files.upload()

Saving dataset.zip.zip to dataset.zip.zip


{'dataset.zip.zip': b'PK\x03\x04\x14\x00\x00\x00\x00\x00\x99[\xa1\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x08\x00 \x00dataset/ux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00UT\r\x00\x07\x1bA\xf4i\x1dC\xf4i\x8b?\xf4iPK\x03\x04\x14\x00\x08\x00\x08\x00\xddZ\xa1\\\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x12\x00 \x00dataset/images.jpgux\x0b\x00\x01\x04\x00\x00\x00\x00\x04\x00\x00\x00\x00UT\r\x00\x07\xba?\xf4i\x02B\xf4i\xb8?\xf4iU\x92\x05P\x14\xd0\xb7\xc6wI\x01\x97\xee%\x16Xra\xa5KP@\xc2%\xa5C\xc2\xa0\xbbSP\x90V\x04\x96\x14\x90\xce\xa5d\x91\x96\xeeXe\x89\xa5\x91\x8e%$DP@\x9e\xfe\xe7\xbdy\xef}g\xce\x9d3\xf7w\xbfs\xcf\xdc\xb97\xb37+\x00jMu\x84:\x00\x08\x04\x00\x80\x7f\x03p\xb3\x00\x88\x04\x90\x91\x90\xd2QS3\xd0R\xff]\xa8\xa9\xa9\x18\xa8\xa8\xa9\xa8\xa8\xff%\x15\x03\x03##\x03\r\x03\x033\xbb\x00\x84\x99\x95\x97\x9d\x81\x81K\x8c\x8bW\x10\x06\x87\xc3\x998%d%Dd\x04D\xe00 999%\x88\x92\x89\x9a\x9a\t\xc6\xce\xce.\xf2\xbf\x82\x89\xfc\x7f\xc1\xfe\x93\xb0\xff\x142\xff\xbd\xf5\xb7

In [48]:
!unzip dataset.zip -d dataset

Archive:  dataset.zip.zip
   creating: dataset/dataset/
  inflating: dataset/dataset/images.jpg  
  inflating: dataset/dataset/img2.jpg  
  inflating: dataset/dataset/img3.jpg  
  inflating: dataset/dataset/img4.jpg  


In [49]:
model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16
).to("cuda")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
!pip install diffusers==0.27.2 transformers accelerate peft==0.11.1 torchao>=0.16.0

In [51]:
!pip uninstall -y torchao peft diffusers transformers accelerate sentence-transformers

!pip install -q \
torchao==0.16.0 \
peft==0.11.1 \
diffusers==0.27.2 \
transformers==4.41.2 \
accelerate==0.29.3

Found existing installation: torchao 0.16.0
Uninstalling torchao-0.16.0:
  Successfully uninstalled torchao-0.16.0
Found existing installation: peft 0.11.1
Uninstalling peft-0.11.1:
  Successfully uninstalled peft-0.11.1
Found existing installation: diffusers 0.27.2
Uninstalling diffusers-0.27.2:
  Successfully uninstalled diffusers-0.27.2
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: accelerate 0.29.3
Uninstalling accelerate-0.29.3:
  Successfully uninstalled accelerate-0.29.3


In [52]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v"],
    lora_dropout=0.1,
    bias="none"
)

pipe.unet = get_peft_model(pipe.unet, lora_config)

In [53]:
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, prompt):
        self.image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir)]
        self.prompt = prompt
        self.tokenizer = pipe.tokenizer

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        image = image.resize((512, 512))

        inputs = self.tokenizer(
            self.prompt,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        )

        return {
            "pixel_values": torch.tensor(image).permute(2, 0, 1) / 255.0,
            "input_ids": inputs.input_ids[0]
        }

In [54]:
import os
print("Current path:", os.getcwd())
print("Files here:", os.listdir())

Current path: /content
Files here: ['.config', 'img3.jpg', 'images (1).jpg', 'dataset', 'img3 (1).jpg', 'img2 (1).jpg', 'dataset.ZIP.zip', 'img4 (1).jpg', 'images (2).jpg', 'dataset.zip.zip', 'images.jpg', 'img2.jpg', '=0.16.0', 'img4.jpg', 'sample_data']


In [46]:
!unzip dataset.zip -d dataset

unzip:  cannot find or open dataset.zip, dataset.zip.zip or dataset.zip.ZIP.


In [55]:
dataset_path = "dataset"   # or "dataset/dataset"

dataset = CustomDataset(dataset_path, prompt="a photo of sks_object")
dataloader = torch.utils.data.DataLoader(dataset, batch_size=2, shuffle=True)

In [57]:
import os

base = "dataset"

while True:
    files = os.listdir(base)
    if len(files) == 1 and os.path.isdir(os.path.join(base, files[0])):
        base = os.path.join(base, files[0])
    else:
        break

print("Using dataset path:", base)

dataset = CustomDataset(base, prompt="a photo of sks_img2")

Using dataset path: dataset/dataset


In [58]:
pipe.save_pretrained("fine_tuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

self.unet=PeftModel(
  (base_model): LoraModel(
    (model): UNet2DConditionModel(
      (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (time_proj): Timesteps()
      (time_embedding): TimestepEmbedding(
        (linear_1): Linear(in_features=320, out_features=1280, bias=True)
        (act): SiLU()
        (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
      )
      (down_blocks): ModuleList(
        (0): CrossAttnDownBlock2D(
          (attentions): ModuleList(
            (0-1): 2 x Transformer2DModel(
              (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
              (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
              (transformer_blocks): ModuleList(
                (0): BasicTransformerBlock(
                  (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
                  (attn1): Attention(
                    (to_q): lora.Linear(
                      (base_layer): Line

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [64]:
import gradio as gr

def generate(prompt):
    image = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality",
        num_inference_steps=40,
        guidance_scale=7.5
    ).images[0]

    return image

demo = gr.Interface(
    fn=generate,
    inputs="text",
    outputs="image",
    title="Custom Image Generator",
    description="Enter a prompt using your trained concept (e.g., sks_img2)"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aa0f3ca31fc5c1f1fe.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
